In [ ]:
import os
import numpy as np
from pathlib import Path
import scipy.ndimage

import scipy.ndimage as ndi

def get_bounding_box(pet_array, ct_array, ct_thresh=-500):
    """
    Step 3 auxiliary function: Maximum 3D bounding box cropping based on connected components.
    To account for baseline shifts (e.g., in ViMedPET), the default ct_thresh is set to -500.
    """
    body_mask = ct_array > ct_thresh
    labels, num_features = ndi.label(body_mask)
    
    if num_features == 0:
        return None
        
    counts = np.bincount(labels.ravel())
    counts[0] = 0  
    largest_component_label = counts.argmax()
    clean_body_mask = (labels == largest_component_label)
    
    z_indices, y_indices, x_indices = np.where(clean_body_mask)
    if len(z_indices) == 0:
        return None
        
    z_min, z_max = z_indices.min(), z_indices.max()
    y_min, y_max = y_indices.min(), y_indices.max()
    x_min, x_max = x_indices.min(), x_indices.max()
    
    pad = 3
    z_min, z_max = max(0, z_min - pad), min(pet_array.shape[0], z_max + pad + 1)
    y_min, y_max = max(0, y_min - pad), min(pet_array.shape[1], y_max + pad + 1)
    x_min, x_max = max(0, x_min - pad), min(pet_array.shape[2], x_max + pad + 1)
    
    return slice(z_min, z_max), slice(y_min, y_max), slice(x_min, x_max)

def normalize_volume(volume, method='z-score'):
    """
    Normalize the data separately.
    The parameter 'method' can be 'z-score' (mean=0, std=1) or 'min-max' (scaled to [0,1]).
    """
    if method == 'z-score':
        mu = np.mean(volume)
        sigma = np.std(volume)
        if sigma == 0:
            return volume - mu
        return (volume - mu) / sigma
    elif method == 'min-max':
        v_min = np.min(volume)
        v_max = np.max(volume)
        if v_max - v_min == 0:
            return volume - v_min
        return (volume - v_min) / (v_max - v_min)
    else:
        raise ValueError("Unsupported normalization method, please choose 'z-score' or 'min-max'")

def process_spade_dataset():
    """
    Complete standalone preprocessing pipeline for the Spade dataset.
    """
    # 1. Define source and target save paths
    source_base = Path('/path/to/your/dataset/spade')
    target_base = Path('/path/to/your/output/PETCTfoundation/Spade')
    
    # Z=3, Y=3.5, X=3.5 (NumPy dimension order is Z, Y, X)
    orig_spacing = np.array([3, 3.5, 3.5])
    target_spacing = np.array([3.0, 2.0, 2.0])
    
    print(f"Starting to process the Spade dataset...")
    print(f"Source directory: {source_base}")
    print(f"Output directory: {target_base}\n")
    
    # 2. Iterate through two levels of folders (PatientID -> ScanID)
    if not source_base.exists():
        print(f"Error: Source folder {source_base} not found.")
        return

    # Iterate through patient level
    for patient_dir in sorted(source_base.iterdir()):
        if not patient_dir.is_dir():
            continue
            
        # Iterate through scan level
        for scan_dir in sorted(patient_dir.iterdir()):
            if not scan_dir.is_dir():
                continue
                
            pet_path = scan_dir / 'pet.npy'
            ct_path = scan_dir / 'ct.npy'
            
            # Process only when both pet.npy and ct.npy exist
            if pet_path.exists() and ct_path.exists():
                try:
                    print(f"Processing data: {patient_dir.name}/{scan_dir.name}")
                    
                    # ---------------- Data Loading and Resampling ----------------
                    pet = np.load(pet_path).astype(np.float32)
                    ct = np.load(ct_path).astype(np.float32)
                    
                    # Step 1: Uniformly space PET to 2mmx2mmx2mm
                    zoom_factors_pet = orig_spacing / target_spacing
                    pet_resampled = scipy.ndimage.zoom(pet, zoom_factors_pet, order=1)
                    
                    # Step 2: Resize CT size to match PET
                    # Use the interpolated PET shape divided by the original CT shape to ensure the final shape is perfectly consistent
                    zoom_factors_ct = np.array(pet_resampled.shape) / np.array(ct.shape)
                    ct_resampled = scipy.ndimage.zoom(ct, zoom_factors_ct, order=1)
                    
                    # Step 3: Remove blank areas to reduce size
                    bbox = get_bounding_box(pet_resampled, ct_resampled, ct_thresh=-500)
                    if bbox is not None:
                        pet_cropped = pet_resampled[bbox]
                        ct_cropped = ct_resampled[bbox]
                    else:
                        pet_cropped, ct_cropped = pet_resampled, ct_resampled   
                        
                    # Step 4: Perform Z-score standardization normalization on CT and PET data separately
                    # Default is z-score here; if mapping to [0, 1] is needed, change 'z-score' to 'min-max'
                    pet_norm = normalize_volume(pet_cropped, method='z-score')
                    ct_norm = normalize_volume(ct_cropped, method='z-score')
                    
                    # Step 2 (cont.): Concatenate PET and CT of the same size into two channels
                    # The final shape is (2, Z, Y, X), Channel 0 is PET, Channel 1 is CT
                    stacked_volume = np.stack([pet_norm, ct_norm], axis=0)
                    
                    # Step 5: Save as an npy file while maintaining the original path structure
                    # Relative path is PatientID/ScanID/pet.npy
                    relative_path = pet_path.relative_to(source_base)
                    save_path = target_base / relative_path
                    
                    # Ensure the folder for the save path exists
                    save_path.parent.mkdir(parents=True, exist_ok=True)
                    
                    # Save and print success message
                    np.save(save_path, stacked_volume)
                    print(f"  -> Successfully saved to: {save_path}")
                    print(f"  -> Final Tensor shape: {stacked_volume.shape}\n")
                    
                except Exception as e:
                    print(f"  -> [Error] An exception occurred while processing {scan_dir}: {e}\n")

if __name__ == '__main__':
    process_spade_dataset()
    print("Spade dataset processing complete.")

In [1]:
import os
from pathlib import Path
import numpy as np

def analyze_dataset_dimensions():
    base_path = Path('/gluon4/xl693/PETCTfoundation/')
    
    if not base_path.exists():
        print(f"Error: Path {base_path} does not exist.")
        return

    print(f"Scanning dataset dimensions in: {base_path}")
    print("=" * 60)
    
    # Iterate through each dataset directory
    for dataset_dir in sorted(base_path.iterdir()):
        if not dataset_dir.is_dir():
            continue
            
        npy_files = list(dataset_dir.rglob('*.npy'))
        if not npy_files:
            continue
            
        # Initialize Min (infinity) and Max (0) for Z, Y, X
        min_z, min_y, min_x = float('inf'), float('inf'), float('inf')
        max_z, max_y, max_x = 0, 0, 0
        
        valid_files = 0
        
        for f in npy_files:
            try:
                # mmap_mode='r' instantly reads ONLY the file header to get the shape
                shape = np.load(f, mmap_mode='r').shape
                
                # Ensure the shape is exactly (Channels, Z, Y, X)
                if len(shape) == 4:
                    c, z, y, x = shape
                    
                    # Update Minimums
                    min_z = min(min_z, z)
                    min_y = min(min_y, y)
                    min_x = min(min_x, x)
                    
                    # Update Maximums
                    max_z = max(max_z, z)
                    max_y = max(max_y, y)
                    max_x = max(max_x, x)
                    
                    valid_files += 1
            except Exception as e:
                print(f"  [Error] Failed reading {f.name}: {e}")
                
        # Print the statistics for this dataset
        if valid_files > 0:
            print(f"Dataset: {dataset_dir.name:<12} | Valid Tensors: {valid_files}")
            print(f"  -> Z-axis (Slice): Min = {min_z:>4}, Max = {max_z:>4}")
            print(f"  -> Y-axis (H)    : Min = {min_y:>4}, Max = {max_y:>4}")
            print(f"  -> X-axis (W)    : Min = {min_x:>4}, Max = {max_x:>4}")
            print("-" * 60)

if __name__ == '__main__':
    analyze_dataset_dimensions()

Scanning dataset dimensions in: /gluon4/xl693/PETCTfoundation
Dataset: AutoPET      | Valid Tensors: 1014
  -> Z-axis (Slice): Min =  200, Max =  661
  -> Y-axis (H)    : Min =  122, Max =  235
  -> X-axis (W)    : Min =  179, Max =  251
------------------------------------------------------------
Dataset: DeepPSMA     | Valid Tensors: 100
  -> Z-axis (Slice): Min =  296, Max =  608
  -> Y-axis (H)    : Min =  151, Max =  293
  -> X-axis (W)    : Min =  196, Max =  369
------------------------------------------------------------
Dataset: Spade        | Valid Tensors: 1137
  -> Z-axis (Slice): Min =    1, Max =  307
  -> Y-axis (H)    : Min =   62, Max =  224
  -> X-axis (W)    : Min =   65, Max =  224
------------------------------------------------------------
Dataset: ViMedPET     | Valid Tensors: 2757
  -> Z-axis (Slice): Min =  207, Max =  551
  -> Y-axis (H)    : Min =  104, Max =  245
  -> X-axis (W)    : Min =  167, Max =  256
----------------------------------------------------

In [4]:
import os
from pathlib import Path
import numpy as np

def analyze_dataset_dimensions():
    base_path = Path('/gluon4/xl693/PETCTfoundation/')
    
    # List of known corrupted/anomalous scans to skip
    anomalies = [
        'LDca4f40/LDca5687',
        'LDca56d5/LDca5e13',
        'LDca4eed/LDca54ed',
        'LDca519f/LDca5602',
        'LDca56da/LDca5d8b',
        'LDca58c7/LDca5c77',
        'LDca58c3/LDca5c6f',
        'LDca4f3f/LDca5507',
        'LDca58c2/LDca5c6e',
        'LDca5163/LDca5581',
        'LDca56d2/LDca5d2b'
    ]
    
    if not base_path.exists():
        print(f"Error: Path {base_path} does not exist.")
        return

    print(f"Scanning dataset dimensions in: {base_path}")
    print(f"Excluding {len(anomalies)} known anomalous samples.")
    print("=" * 60)
    
    # Iterate through each dataset directory
    for dataset_dir in sorted(base_path.iterdir()):
        if not dataset_dir.is_dir():
            continue
            
        npy_files = list(dataset_dir.rglob('*.npy'))
        if not npy_files:
            continue
            
        # Initialize Min (infinity) and Max (0) for Z, Y, X
        min_z, min_y, min_x = float('inf'), float('inf'), float('inf')
        max_z, max_y, max_x = 0, 0, 0
        
        valid_files = 0
        
        for f in npy_files:
            # Check if the current file path contains any of the anomaly strings
            if any(anomaly in f.as_posix() for anomaly in anomalies):
                continue  # Skip this anomalous file
                
            try:
                # mmap_mode='r' instantly reads ONLY the file header to get the shape
                shape = np.load(f, mmap_mode='r').shape
                
                # Ensure the shape is exactly (Channels, Z, Y, X)
                if len(shape) == 4:
                    c, z, y, x = shape
                    
                    # Update Minimums
                    min_z = min(min_z, z)
                    min_y = min(min_y, y)
                    min_x = min(min_x, x)
                    
                    # Update Maximums
                    max_z = max(max_z, z)
                    max_y = max(max_y, y)
                    max_x = max(max_x, x)
                    
                    valid_files += 1
            except Exception as e:
                print(f"  [Error] Failed reading {f.name}: {e}")
                
        # Print the statistics for this dataset
        if valid_files > 0:
            print(f"Dataset: {dataset_dir.name:<12} | Valid Tensors: {valid_files}")
            print(f"  -> Z-axis (Slice): Min = {min_z:>4}, Max = {max_z:>4}")
            print(f"  -> Y-axis (H)    : Min = {min_y:>4}, Max = {max_y:>4}")
            print(f"  -> X-axis (W)    : Min = {min_x:>4}, Max = {max_x:>4}")
            print("-" * 60)

if __name__ == '__main__':
    analyze_dataset_dimensions()

Scanning dataset dimensions in: /gluon4/xl693/PETCTfoundation
Excluding 11 known anomalous samples.
Dataset: AutoPET      | Valid Tensors: 1014
  -> Z-axis (Slice): Min =  200, Max =  661
  -> Y-axis (H)    : Min =  122, Max =  235
  -> X-axis (W)    : Min =  179, Max =  251
------------------------------------------------------------
Dataset: DeepPSMA     | Valid Tensors: 100
  -> Z-axis (Slice): Min =  296, Max =  608
  -> Y-axis (H)    : Min =  151, Max =  293
  -> X-axis (W)    : Min =  196, Max =  369
------------------------------------------------------------
Dataset: Spade        | Valid Tensors: 1126
  -> Z-axis (Slice): Min =  103, Max =  307
  -> Y-axis (H)    : Min =  109, Max =  224
  -> X-axis (W)    : Min =  141, Max =  224
------------------------------------------------------------
Dataset: ViMedPET     | Valid Tensors: 2757
  -> Z-axis (Slice): Min =  207, Max =  551
  -> Y-axis (H)    : Min =  104, Max =  245
  -> X-axis (W)    : Min =  167, Max =  256
--------------